<a href="https://colab.research.google.com/github/Rishii077/AI-Lab-Assignments/blob/main/Experiment_2_RAG_QA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Create a sample document for our RAG system

document_text = """
Artificial Intelligence (AI) is a branch of computer science that focuses on
creating systems capable of performing tasks that normally require human
intelligence.

Machine Learning (ML) is a subset of Artificial Intelligence. It allows
computers to learn patterns from data and make predictions or decisions
without being explicitly programmed for every task.

Deep Learning is a subset of Machine Learning that uses artificial neural
networks with multiple layers. It is commonly used for image recognition,
speech recognition, natural language processing, and other complex tasks.

Natural Language Processing (NLP) is a field of Artificial Intelligence that
enables computers to understand, process, and generate human language.
Applications of NLP include chatbots, translation systems, sentiment analysis,
and text summarization.

Generative AI refers to artificial intelligence systems that can create new
content such as text, images, audio, video, and code. Large Language Models
(LLMs) are an important type of generative AI system designed to understand
and generate human language.

Retrieval-Augmented Generation (RAG) combines information retrieval with
generative AI. Instead of relying only on information learned during model
training, a RAG system retrieves relevant information from an external
knowledge source and provides that information to a language model to
generate a more relevant answer.
"""

# Save the document as a text file

with open("ai_knowledge.txt", "w") as file:
    file.write(document_text)

print("Document created successfully!")

Document created successfully!


In [2]:
# Read the document

with open("ai_knowledge.txt", "r") as file:
    document = file.read()

print(document)


Artificial Intelligence (AI) is a branch of computer science that focuses on
creating systems capable of performing tasks that normally require human
intelligence.

Machine Learning (ML) is a subset of Artificial Intelligence. It allows
computers to learn patterns from data and make predictions or decisions
without being explicitly programmed for every task.

Deep Learning is a subset of Machine Learning that uses artificial neural
networks with multiple layers. It is commonly used for image recognition,
speech recognition, natural language processing, and other complex tasks.

Natural Language Processing (NLP) is a field of Artificial Intelligence that
enables computers to understand, process, and generate human language.
Applications of NLP include chatbots, translation systems, sentiment analysis,
and text summarization.

Generative AI refers to artificial intelligence systems that can create new
content such as text, images, audio, video, and code. Large Language Models
(LLMs) are

In [3]:
# Split the document into smaller chunks

def create_chunks(text, chunk_size=500):

    chunks = []

    for i in range(0, len(text), chunk_size):
        chunk = text[i:i + chunk_size]
        chunks.append(chunk)

    return chunks


chunks = create_chunks(document)

print("Number of chunks:", len(chunks))

for i, chunk in enumerate(chunks):
    print("\n--- Chunk", i + 1, "---")
    print(chunk)

Number of chunks: 3

--- Chunk 1 ---

Artificial Intelligence (AI) is a branch of computer science that focuses on
creating systems capable of performing tasks that normally require human
intelligence.

Machine Learning (ML) is a subset of Artificial Intelligence. It allows
computers to learn patterns from data and make predictions or decisions
without being explicitly programmed for every task.

Deep Learning is a subset of Machine Learning that uses artificial neural
networks with multiple layers. It is commonly used for image re

--- Chunk 2 ---
cognition,
speech recognition, natural language processing, and other complex tasks.

Natural Language Processing (NLP) is a field of Artificial Intelligence that
enables computers to understand, process, and generate human language.
Applications of NLP include chatbots, translation systems, sentiment analysis,
and text summarization.

Generative AI refers to artificial intelligence systems that can create new
content such as text, images, a

In [4]:
print("Total chunks created:", len(chunks))

Total chunks created: 3


In [5]:
!pip install -q -U google-genai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 39.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 262.4/262.4 kB 15.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.58.0 which is incompatible.


In [7]:
from google.colab import userdata
from google import genai

API_KEY = userdata.get("GEMINI_API_KEY")

client = genai.Client(api_key=API_KEY)

print("Gemini connected successfully!")

Gemini connected successfully!


In [8]:
# Create embeddings for each document chunk

embeddings = []

for chunk in chunks:

    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=chunk
    )

    embeddings.append(result.embeddings[0].values)

print("Number of embeddings created:", len(embeddings))
print("Size of first embedding:", len(embeddings[0]))

Number of embeddings created: 3
Size of first embedding: 3072


In [9]:
import numpy as np

def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)

    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


def retrieve_relevant_chunk(question):

    # Create embedding for the question
    result = client.models.embed_content(
        model="gemini-embedding-001",
        contents=question
    )

    question_embedding = result.embeddings[0].values

    # Calculate similarity with every document chunk
    similarities = []

    for i, embedding in enumerate(embeddings):
        score = cosine_similarity(question_embedding, embedding)
        similarities.append((i, score))

    # Find the most relevant chunk
    best_chunk_index, best_score = max(
        similarities,
        key=lambda x: x[1]
    )

    return chunks[best_chunk_index], best_score

In [10]:
question = "What is Machine Learning?"

relevant_chunk, score = retrieve_relevant_chunk(question)

print("Question:")
print(question)

print("\nMost Relevant Chunk:")
print(relevant_chunk)

print("\nSimilarity Score:")
print(score)

Question:
What is Machine Learning?

Most Relevant Chunk:

Artificial Intelligence (AI) is a branch of computer science that focuses on
creating systems capable of performing tasks that normally require human
intelligence.

Machine Learning (ML) is a subset of Artificial Intelligence. It allows
computers to learn patterns from data and make predictions or decisions
without being explicitly programmed for every task.

Deep Learning is a subset of Machine Learning that uses artificial neural
networks with multiple layers. It is commonly used for image re

Similarity Score:
0.6686559049223958


In [11]:
def rag_question_answer(question):

    # Retrieve the most relevant chunk
    relevant_chunk, score = retrieve_relevant_chunk(question)

    # Create prompt using retrieved information
    prompt = f"""
You are a helpful AI assistant.

Answer the user's question using ONLY the information
provided in the context below.

Context:
{relevant_chunk}

User Question:
{question}

Give a clear and simple answer.
If the answer is not available in the context, say:
"Information not available in the provided document."
"""

    # Generate answer using Gemini
    interaction = client.interactions.create(
        model="gemini-3.6-flash",
        input=prompt
    )

    answer = interaction.output_text.strip()

    print("Question:")
    print(question)

    print("\nRetrieved Context:")
    print(relevant_chunk)

    print("\nAnswer:")
    print(answer)

In [12]:
rag_question_answer("What is Machine Learning?")

Question:
What is Machine Learning?

Retrieved Context:

Artificial Intelligence (AI) is a branch of computer science that focuses on
creating systems capable of performing tasks that normally require human
intelligence.

Machine Learning (ML) is a subset of Artificial Intelligence. It allows
computers to learn patterns from data and make predictions or decisions
without being explicitly programmed for every task.

Deep Learning is a subset of Machine Learning that uses artificial neural
networks with multiple layers. It is commonly used for image re

Answer:
Based on the provided context, Machine Learning (ML) is a subset of Artificial Intelligence that allows computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every task.


In [13]:
rag_question_answer("What are the applications of NLP?")

Question:
What are the applications of NLP?

Retrieved Context:
cognition,
speech recognition, natural language processing, and other complex tasks.

Natural Language Processing (NLP) is a field of Artificial Intelligence that
enables computers to understand, process, and generate human language.
Applications of NLP include chatbots, translation systems, sentiment analysis,
and text summarization.

Generative AI refers to artificial intelligence systems that can create new
content such as text, images, audio, video, and code. Large Language Models
(LLMs) are

Answer:
Based on the provided context, the applications of NLP include:

* Chatbots
* Translation systems
* Sentiment analysis
* Text summarization
